# CS 3892 / 5892 — Session 26 · AI for formal methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-12-03-ai-for-formal-methods.ipynb)

**Thursday, December 3, 2026.** An LLM writes the specification, or the proof, and a tool decides.
Two real runs of Claude Haiku 4.5, saved verbatim in `llm/` (no tools, an empty directory, one
attempt each; `llm/run.md`): four English requirements to **LTL**, judged by **NuSMV** three ways
(syntax, truth, meaning); and a **Lean** proof over two rounds, judged by Lean. No model is called
here: the notebook checks the saved replies.

## Setup

Run these two once. The second installs Lean (HW4's installer); NuSMV is fetched by the first script.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-12-03-ai-for-formal-methods"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

PYD = ROOT / "sessions" / SESSION / "python"
LD  = ROOT / "sessions" / SESSION / "lean"
SMVD = ROOT / "sessions" / SESSION / "smv"
LLMD = ROOT / "sessions" / SESSION / "llm"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- Lean v4.33.0, installed by HW4's checker: elan (the Lean version manager, pinned by
# SHA-256) into ~/.elan, then Lean through it -- about 575 MB, a minute or two the first time,
# and nothing to install after that. A Lean you already have is used as it is.
out = run(PYD / "lean_cli.py")
assert "version 4.33.0" in out

## 1. English → LTL, and three checks

The prompt, then the checks: does NuSMV accept the lines; does each formula hold on the counter;
is each the same property as a reference (on a free model)?

In [ ]:
display(SVG(filename=str(FIG / "counter.svg"))); display(SVG(filename=str(FIG / "free_x.svg")))
print((LLMD / "ltl_prompt.txt").read_text())
show(SMVD / "counter.smv"); show(SMVD / "free.smv"); show(PYD / "nusmv.py")
out = run(PYD / "01_llm_ltl.py")
assert "only the third caught R4" in out

## 2. A Lean proof, two rounds, Lean as the judge

In [ ]:
print((LLMD / "lean_prompt_round1.txt").read_text())
out = run(PYD / "02_llm_lean.py")
assert "the checker, not the model, decides" in out

## Try it yourself

- Rewrite R4 in English so that it can only mean one of the two readings. Which words did you need?
- Give the round-2 error to an assistant yourself (`llm/lean_prompt_round2.txt` plus Lean's
  message). Does round 3 compile? Check it with `lean_cli.messages`.
- Write R5, "the counter never skips a value below 10", as LTL. Check it against the counter and
  against a reference of your own.